## Query Enhancement

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS


#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

C:\Users\kanha\AppData\Local\Temp\ipykernel_20508\3387885773.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
## Step 1:vectorstore and Load and split

loader=TextLoader("langchain_crewai_dataset.txt")
raw_docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(raw_docs)

In [3]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [4]:
### Step 2:vectorstore
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
vectorstore=FAISS.from_documents(chunks,embeddings)
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":8})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023EA997AB90>, search_type='mmr', search_kwargs={'k': 8})

In [7]:
## Step 4: LLM and prompt
import os
from dotenv import load_dotenv
load_dotenv()
API=os.getenv("OPENAI_API_KEY")
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key=API)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023EA9FEA0D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023EE4E3C150>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
query_expansion_Prompt=PromptTemplate.from_template("""
You are a helpful asisstant . Expand the following query to improve document retrieval by adding relevant synonyms and document terms.
                                                    
Original query:"{query}"
Expanded Query:                                                    
                                                
""")
query_expansion_chain=query_expansion_Prompt|llm|StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful asisstant . Expand the following query to improve document retrieval by adding relevant synonyms and document terms.\n\nOriginal query:"{query}"\nExpanded Query:                                                    \n\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023EA9FEA0D0>, async_client=<groq.

In [9]:
query_expansion_chain.invoke({"query":"langchain-memory"})

'To improve document retrieval, I\'ll add relevant synonyms and document terms to the original query. Here\'s the expanded query:\n\nExpanded Query: ("langchain-memory" OR "language model memory" OR "LLaMA memory" OR "large language model recall" OR "conversational AI memory" OR "long-term memory in language models" OR "knowledge retention in LLMs" OR "memory-augmented language models")\n\nThis expanded query includes:\n\n1. Synonyms: "language model memory", "LLaMA memory" (referring to the LLaMA model)\n2. Related concepts: "large language model recall", "conversational AI memory"\n3. Specific document terms: "long-term memory in language models", "knowledge retention in LLMs"\n4. Relevant technologies: "memory-augmented language models"\n\nBy incorporating these additional terms, the expanded query should improve the retrieval of relevant documents related to the original topic of "langchain-memory".'

In [10]:
answer_prmpt=PromptTemplate.from_template("""
Answer the question based on the context below:
                                          
Context:
{context}
Question:{input}                         
                                                           """)
document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prmpt)

rag_pipelinr=(
    RunnableMap({
        "input":lambda x:x['input'],
        "context":lambda x: retriever.invoke(query_expansion_chain.invoke({"query":x['input']}))
    })
    | document_chain
)


In [11]:
query={"input":"What types of memory does Langchain support?"}
print(query_expansion_chain.invoke({"query":query}))
response=rag_pipelinr.invoke(query)
print("... Answer:\n",response)

To improve document retrieval, the original query can be expanded by incorporating relevant synonyms and document terms related to "memory" and "Langchain." Here's an expanded version:

Expanded Query: 
"{'input': 'What types of memory does Langchain support, including RAM, storage, cache, and virtual memory? Are there any specific memory architectures, such as volatile or non-volatile memory, that Langchain is compatible with? Does Langchain have any memory management capabilities, like memory allocation or deallocation, and how does it handle memory constraints or limitations?'}

Alternatively, a more concise version that still includes relevant synonyms and terms could be:

Expanded Query: 
"{'input': 'What memory types, such as RAM, storage, or cache, are supported by Langchain, and what are its memory management capabilities?'}

This expanded query aims to capture a broader range of relevant documents by including terms that are synonymous with "memory" in the context of computing